<a href="https://colab.research.google.com/github/aliaksandra-babova/ML-zoomcamp-homework/blob/main/Homework_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
!unzip data.zip

--2025-12-01 17:41:07--  https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/405934815/e712cf72-f851-44e0-9c05-e711624af985?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-01T18%3A15%3A29Z&rscd=attachment%3B+filename%3Ddata.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-01T17%3A14%3A59Z&ske=2025-12-01T18%3A15%3A29Z&sks=b&skv=2018-11-09&sig=emMxo%2B%2FkMYVAk5f29NytAWylx5rqNLwAkU9qF9GdEzY%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2NDYxMjY2NywibmJmIjoxNzY0NjEwODY3LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdG

In [10]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
class HairStyleDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            label_dir = os.path.join(data_dir, label_name)
            for img_name in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
input_size = 200

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [5]:
train_dataset = HairStyleDataset(
    data_dir='/content/data/train',
    transform=train_transforms
)

val_dataset = HairStyleDataset(
    data_dir='/content/data/train',
    transform=val_transforms
)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=20, shuffle=False)

In [6]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=(3,3))
        self.relu1 = nn.ReLU()

        self.pool1 = nn.MaxPool2d(kernel_size=(2,2))

        self.flatten = nn.Flatten()

        # Calculate flattened size for 200x200 input:
        # After Conv2d(3, 32, 3x3): (200 - 3 + 1) = 198 -> (32, 198, 198)
        # After MaxPool2d(2x2): 198 / 2 = 99 -> (32, 99, 99)
        # Flattened size: 32 * 99 * 99 = 313632

        self.dense1 = nn.Linear(in_features=313632, out_features=64)
        self.relu2 = nn.ReLU()

        self.dense2 = nn.Linear(in_features=64, out_features=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.flatten(x)
        x = self.dense1(x)
        x = self.relu2(x)
        x = self.dense2(x)
        return x

# Instantiate the PyTorch model
model = SimpleCNN()

optimizer = optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

loss_fn = nn.BCEWithLogitsLoss()

print("PyTorch model, optimizer, and loss function initialized successfully.")


PyTorch model, optimizer, and loss function initialized successfully.


In [8]:
# Define device here to ensure it's available for torchsummary
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the correct device BEFORE calling summary
model.to(device)

summary(model, input_size=(3, 200, 200), device=str(device))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
              ReLU-2         [-1, 32, 198, 198]               0
         MaxPool2d-3           [-1, 32, 99, 99]               0
           Flatten-4               [-1, 313632]               0
            Linear-5                   [-1, 64]      20,072,512
              ReLU-6                   [-1, 64]               0
            Linear-7                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 23.93
Params size (MB): 76.57
Estimated Total Size (MB): 100.96
----------------------------------------------------------------


In [9]:
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion = loss_fn
validation_loader = val_loader
validation_dataset = val_dataset

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6552, Acc: 0.6280, Val Loss: 0.5997, Val Acc: 0.6642
Epoch 2/10, Loss: 0.5715, Acc: 0.6929, Val Loss: 0.5084, Val Acc: 0.7441
Epoch 3/10, Loss: 0.5238, Acc: 0.7228, Val Loss: 0.6079, Val Acc: 0.6317
Epoch 4/10, Loss: 0.5835, Acc: 0.6991, Val Loss: 0.5191, Val Acc: 0.7316
Epoch 5/10, Loss: 0.5166, Acc: 0.7541, Val Loss: 0.5933, Val Acc: 0.6916
Epoch 6/10, Loss: 0.4594, Acc: 0.7753, Val Loss: 0.5429, Val Acc: 0.7228
Epoch 7/10, Loss: 0.3950, Acc: 0.8165, Val Loss: 0.7827, Val Acc: 0.6130
Epoch 8/10, Loss: 0.3874, Acc: 0.8252, Val Loss: 0.7066, Val Acc: 0.7129
Epoch 9/10, Loss: 0.3170, Acc: 0.8689, Val Loss: 0.1778, Val Acc: 0.9451
Epoch 10/10, Loss: 0.2316, Acc: 0.9014, Val Loss: 0.6763, Val Acc: 0.6854


In [14]:
accuracy = pd.Series(history['acc'])
accuracy.median()

0.764669163545568

In [15]:
train_loss = pd.Series(history['loss'])
train_loss.std()

0.13122644385674692

In [16]:
train_transforms = transforms.Compose([
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [17]:
train_dataset = HairStyleDataset(
    data_dir='/content/data/train',
    transform=train_transforms
)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)

In [18]:
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion = loss_fn
validation_loader = val_loader
validation_dataset = val_dataset

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6593, Acc: 0.6604, Val Loss: 0.5310, Val Acc: 0.7091
Epoch 2/10, Loss: 0.5865, Acc: 0.6904, Val Loss: 0.3758, Val Acc: 0.8065
Epoch 3/10, Loss: 0.5464, Acc: 0.7203, Val Loss: 0.4206, Val Acc: 0.7953
Epoch 4/10, Loss: 0.5117, Acc: 0.7291, Val Loss: 0.4802, Val Acc: 0.7316
Epoch 5/10, Loss: 0.5084, Acc: 0.7516, Val Loss: 0.3690, Val Acc: 0.8277
Epoch 6/10, Loss: 0.4886, Acc: 0.7690, Val Loss: 0.4882, Val Acc: 0.7303
Epoch 7/10, Loss: 0.4798, Acc: 0.7615, Val Loss: 0.4936, Val Acc: 0.7266
Epoch 8/10, Loss: 0.4710, Acc: 0.7653, Val Loss: 0.4062, Val Acc: 0.8040
Epoch 9/10, Loss: 0.4710, Acc: 0.7815, Val Loss: 0.3362, Val Acc: 0.8652
Epoch 10/10, Loss: 0.4747, Acc: 0.7628, Val Loss: 0.4491, Val Acc: 0.7853


In [19]:
val_loss = pd.Series(history['val_loss'])
val_loss.mean()

np.float64(0.43498405793521283)

In [20]:
val_accuracy = pd.Series(history['val_acc'][5:])
val_accuracy.mean()

np.float64(0.7822721598002497)